# EDA — Retail Basket Dynamics (cleaning stage)

**Purpose:** connect to the DB, load the cleaned SQL view, inspect structure, and perform initial cleaning (duplicates, types, basic features).  
Each code cell below is followed by a short note explaining why it exists and what to check.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# Connect to SQLite via SQLAlchemy
engine = create_engine("sqlite:///retail.db")

print("Connection ready.")

Connection ready.


### Why connect here
We keep all data in `retail.db`. This cell creates a reusable `engine` object so every subsequent cell queries the same source. We do not read Excel here.

In [ ]:
# Confirm tables and views exist
with engine.connect() as conn:
    tables = conn.execute(text(
        "SELECT name FROM sqlite_master WHERE type='table'"
    )).fetchall()
    
    views = conn.execute(text(
        "SELECT name FROM sqlite_master WHERE type='view'"
    )).fetchall()

print("Tables:", tables)
print("Views:", views)

Tables: [('transactions_raw',)]
Views: [('transactions_clean',)]


In [ ]:
# Load the cleaned SQL view into pandas DataFrame
query = """
SELECT * FROM transactions_clean;
"""

df = pd.read_sql(query, engine)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
# Quick look and basic counts
print(f"Shape: {df.shape}\nDuplcate Rows: {df.duplicated().sum()}")
df.isnull().sum()

Shape: (805549, 8)
Duplcate Rows: 26124


Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [29]:
# before removing duplicates
df['Invoice'].nunique()

36969

In [30]:
df.drop_duplicates().shape[0]

779425

In [31]:
# after removing duplicates
df['Invoice'].nunique()

36969

In [37]:
df = df.drop_duplicates()
df.shape

(779425, 8)

### Step 5: Check unique invoices before and after removing duplicates

We compare the number of unique invoice numbers before and after dropping duplicates.

Reason:
If unique invoice count remains the same, it means duplicate rows do NOT represent separate business transactions and are safe to remove.

If invoice count changed, it would indicate the duplicates had business meaning and should be handled more carefully.

Here the unique count is the same before and after removing duplicates.

In [ ]:
#  Checking data types of each column
df.dtypes

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

In [39]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

bad_dates = df['InvoiceDate'].isna().sum()
print("Failed InvoiceDate parses (NaT):", bad_dates)

Failed InvoiceDate parses (NaT): 0


`InvoiceDate` was stored as text, so converted it to a true datetime type and counted any rows that failed to parse. Using `errors='coerce'` replaces invalid formats with `NaT`, allowing the pipeline to continue while revealing data quality problems. This step ensures all downstream time-based analysis relies on consistent datetime values.

In [41]:
df['Customer ID'] = df['Customer ID'].astype('Int64')

In [42]:
print("\nAfter conversion dtypes:")
df.dtypes


After conversion dtypes:


Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID             Int64
Country                object
dtype: object

In [43]:
df['Description'] = df['Description'].astype(str).str.strip()
df['StockCode'] = df['StockCode'].astype(str).str.strip()

Applied light cleaning to the `Description` and `StockCode` fields by stripping whitespace and enforcing string type. This prevents subtle grouping errors during product level analysis, such as the same item appearing as multiple products due to trailing spaces or inconsistent formatting.  

In [45]:
df['TotalPrice'] = df['Quantity'] * df['Price']


##### Creating the TotalPrice (Revenue) Feature

`TotalPrice` represents the revenue contribution of each line item.  
This is the core monetary feature used in almost all downstream analysis including product performance, customer value, basket size, and country-level revenue.  
We can only confirm data quality (e.g., no negative revenues) after creating this column.

In [ ]:
print("Qty <= 0:", df[df['Quantity'] <= 0].shape[0])
print("Price <= 0:", df[df['Price'] <= 0].shape[0])
print("TotalPrice <= 0:", df[df['TotalPrice'] <= 0].shape[0])

Qty <= 0: 0
Price <= 0: 0
TotalPrice <= 0: 0


In [47]:
print(df[['Quantity','Price','TotalPrice']].head())

   Quantity  Price  TotalPrice
0        12   6.95        83.4
1        12   6.75        81.0
2        12   6.75        81.0
3        48   2.10       100.8
4        24   1.25        30.0


In [49]:
df['Country'].unique()

array(['United Kingdom', 'France', 'USA', 'Belgium', 'Australia', 'EIRE',
       'Germany', 'Portugal', 'Denmark', 'Netherlands', 'Poland',
       'Channel Islands', 'Spain', 'Cyprus', 'Greece', 'Norway',
       'Austria', 'Sweden', 'United Arab Emirates', 'Finland', 'Italy',
       'Switzerland', 'Japan', 'Unspecified', 'Nigeria', 'Malta', 'RSA',
       'Singapore', 'Bahrain', 'Thailand', 'Israel', 'Lithuania',
       'West Indies', 'Korea', 'Brazil', 'Canada', 'Iceland', 'Lebanon',
       'Saudi Arabia', 'Czech Republic', 'European Community'],
      dtype=object)

### Dataset is now ready for feature engineering
The dataset is now:

Structurally clean                          
Duplicate-free                          
Correct dtypes                          
No missing critical values                          
Valid monetary values                           
Valid dates                         
Clean country values                            
Ready for analysis

In [53]:
# Time-based feature creation

df['InvoiceDay'] = df['InvoiceDate'].dt.date               # daily grain
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')   # monthly grain
df['InvoiceWeek'] = df['InvoiceDate'].dt.isocalendar().week # weekly grain
df['InvoiceHour'] = df['InvoiceDate'].dt.hour              # hour of day
df['InvoiceMonthName'] = df['InvoiceDate'].dt.month_name()

df[['InvoiceDate','InvoiceDay','InvoiceMonth','InvoiceWeek','InvoiceHour', 'InvoiceMonthName' ]].head()

,InvoiceDate,InvoiceDay,InvoiceMonth,InvoiceWeek,InvoiceHour,InvoiceMonthName
0,2009-12-01 07:45:00,2009-12-01,2009-12,49,7,December
1,2009-12-01 07:45:00,2009-12-01,2009-12,49,7,December
2,2009-12-01 07:45:00,2009-12-01,2009-12,49,7,December
3,2009-12-01 07:45:00,2009-12-01,2009-12,49,7,December
4,2009-12-01 07:45:00,2009-12-01,2009-12,49,7,December


### Creating Time-Based Features

Extracted multiple temporal components from `InvoiceDate` because retail behaviour is strongly time dependent.  
Each feature supports a different level of time-series aggregation:

- **InvoiceDay** enables daily revenue trends.  
- **InvoiceMonth** (as a Period) supports correct monthly grouping and chronological sorting for time-series analysis.  
- **InvoiceWeek** captures weekly seasonal patterns common in retail operations.  
- **InvoiceHour** reveals intraday shopping behaviour and operational load.  
- **InvoiceMonthName** provides readable month labels for later visualizations.

Using both analytical (Period) and presentation-friendly (MonthName) representations preserves accuracy during analysis and clarity during reporting.

In [ ]:
# INVOICE / BASKET FEATURES

df['InvoiceTotal'] = df.groupby('Invoice')['TotalPrice'].transform('sum')

df['InvoiceTotalQuantity'] = df.groupby('Invoice')['Quantity'].transform('sum')

df['InvoiceLineCount'] = df.groupby('Invoice')['StockCode'].transform('count')

df['InvoiceUniqueProducts'] = df.groupby('Invoice')['StockCode'].transform('nunique')

df['InvoiceAvgPricePerUnit'] = df['InvoiceTotal'] / df['InvoiceTotalQuantity']

df.loc[df['InvoiceTotalQuantity'] == 0, 'InvoiceAvgPricePerUnit'] = pd.NA
# Quick checks
print("Feature columns added. Data shape:", df.shape)
display(df[['Invoice','InvoiceTotal','InvoiceTotalQuantity','InvoiceLineCount','InvoiceUniqueProducts','InvoiceAvgPricePerUnit']].drop_duplicates().head(10))

Feature columns added. Data shape: (779425, 19)


,Invoice,InvoiceTotal,InvoiceTotalQuantity,InvoiceLineCount,InvoiceUniqueProducts,InvoiceAvgPricePerUnit
0,489434,505.30,166,8,8,3.043976
8,489435,145.80,60,4,4,2.430000
12,489436,630.33,193,19,19,3.265959
31,489437,310.75,145,23,23,2.143103
54,489438,2286.24,826,17,17,2.767845
71,489439,426.30,219,19,19,1.946575
90,489440,50.40,16,2,2,3.150000
92,489441,344.34,102,4,4,3.375882
96,489442,382.37,275,23,23,1.390436
119,489443,285.06,120,7,7,2.375500


### Invoice / Basket-Level Features

Created basket level metrics by aggregating each invoice across its line items.  
These features describe the structure and value of each shopping session and help answer questions about basket size, diversity, and spending behaviour.

- **InvoiceTotal**: total revenue from the invoice. Captures full basket value, including returns if present.  
- **InvoiceTotalQuantity**: total units purchased in the invoice, useful for understanding volume per basket.  
- **InvoiceLineCount**: number of line items; differs from quantity because a single line can contain multiple units.  
- **InvoiceUniqueProducts**: number of distinct products purchased in the invoice, indicating basket diversity.  
- **InvoiceAvgPricePerUnit**: simple price-per-unit signal for the invoice, used to compare low value vs high value baskets.

These basket features support analysis of purchasing patterns, cross-sell potential, and operational load by showing what typical transactions look like at the session level.

In [56]:
# CUSTOMER FEATURES

df['CustomerTotalSpend'] = df.groupby('Customer ID')['TotalPrice'].transform('sum')

df['CustomerNumPurchases'] = df.groupby('Customer ID')['Invoice'].transform('nunique')

df['CustomerUniqueProducts'] = df.groupby('Customer ID')['StockCode'].transform('nunique')

# check a few
df[['Customer ID','CustomerTotalSpend','CustomerNumPurchases','CustomerUniqueProducts']].head()

,Customer ID,CustomerTotalSpend,CustomerNumPurchases,CustomerUniqueProducts
0,13085,2433.28,8,50
1,13085,2433.28,8,50
2,13085,2433.28,8,50
3,13085,2433.28,8,50
4,13085,2433.28,8,50


### Customer-Level Features

Created customer-level spend and activity metrics to understand purchasing behaviour.  
- **CustomerTotalSpend** shows how much each customer has spent in total.  
- **CustomerNumPurchases** counts how many separate orders the customer made.  
- **CustomerUniqueProducts** captures how many different products the customer has bought.  

These features help identify high-value customers, repeat buyers, and customers with wide or narrow shopping patterns.

In [58]:
# PRODUCT FEATURES

# 1. Total revenue for each product
df['ProductTotalRevenue'] = df.groupby('StockCode')['TotalPrice'].transform('sum')

# 2. Total quantity sold for each product
df['ProductTotalQuantity'] = df.groupby('StockCode')['Quantity'].transform('sum')

# 3. Number of unique customers who bought the product
df['ProductNumCustomers'] = df.groupby('StockCode')['Customer ID'].transform('nunique')

df[['StockCode','ProductTotalRevenue','ProductTotalQuantity','ProductNumCustomers']].head()

,StockCode,ProductTotalRevenue,ProductTotalQuantity,ProductNumCustomers
0,85048,18618.75,2525,309
1,79323P,13198.10,2234,132
2,79323W,17384.80,2948,174
3,22041,15088.77,6845,101
4,21232,40994.71,34935,689


### Product-Level Features

Created revenue and demand metrics for each product.  
- **ProductTotalRevenue** shows how much money each product generated.  
- **ProductTotalQuantity** tells how many units were sold.  
- **ProductNumCustomers** counts how many different customers purchased the product.  

These features help identify top sellers, underperforming products, and items with broad or narrow customer appeal.

In [61]:
# COUNTRY-LEVEL FEATURES

# 1. Total revenue per country
df['CountryTotalRevenue'] = df.groupby('Country')['TotalPrice'].transform('sum')

# 2. Number of unique customers per country
df['CountryCustomerCount'] = df.groupby('Country')['Customer ID'].transform('nunique')

# 3. Number of invoices per country
df['CountryInvoiceCount'] = df.groupby('Country')['Invoice'].transform('nunique')

# Preview
df[['Country','CountryTotalRevenue','CountryCustomerCount','CountryInvoiceCount']].head()

,Country,CountryTotalRevenue,CountryCustomerCount,CountryInvoiceCount
0,United Kingdom,1.438923e+07,5350,33541
1,United Kingdom,1.438923e+07,5350,33541
2,United Kingdom,1.438923e+07,5350,33541
3,United Kingdom,1.438923e+07,5350,33541
4,United Kingdom,1.438923e+07,5350,33541


### Country-Level Features

Added country-level metrics to compare performance across regions.  
- **CountryTotalRevenue** shows total revenue contributed by each country.  
- **CountryCustomerCount** measures how many unique customers come from each country.  
- **CountryInvoiceCount** counts how many orders were placed in each region.  

These features help identify strong and weak markets and support geographic analysis of customer activity and revenue performance.

### Return Features Removed

Verified that this dataset contains no negative quantities and therefore no return transactions.  
Return-related columns were removed to avoid unnecessary complexity and keep the dataset focused on valid purchasing behaviour.

In [64]:
# Save as CSV
df.to_csv("retail_features.csv", index=False)
print("Files saved: retail_features.csv")

Files saved: retail_features.csv
